In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
DIR = Path("data")

for subdir in DIR.iterdir():
    if subdir.is_dir():
        print(f"--- Traitement du dossier {subdir.name} ---")
        csv_files = sorted(subdir.glob("*.csv"))

        cleaned_dataframes = {}

        for file_path in csv_files:
            print(f"--- Traitement de {file_path.name} ---")

            df = pd.read_csv(file_path)
            print(f"shape: {df.shape}\n")

            # supression des doublons
            nb_doublons = df.duplicated().sum()
            if nb_doublons > 0:
                print(f"{nb_doublons} ligne dupliquée trouvée")
                df = df.drop_duplicates()

            #supression colonnes inutiles
            for col in df.columns:
                if df[col].nunique() == 1:
                    print(f"Colonne '{col}' supprimée car elle contient une seule valeur unique")
                    df = df.drop(columns=[col])

            for col in df.columns:
                if df[col].isnull().sum() / len(df) > 0.5:
                    print(f"Colonne '{col}' supprimée car elle contient plus de 50% de valeurs nulles")
                    df = df.drop(columns=[col])

            #supression des valeurs nulles
            nb_nulls = df.isnull().sum().sum()
            if nb_nulls > 0:
                print(f"{nb_nulls} valeurs nulles trouvées")
                df = df.dropna()

            #supression des valeurs abérrantes basé sur l'écart interquartile
            numeric_cols = df.select_dtypes(include=['number']).columns
            for col in numeric_cols:
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
                if not outliers.empty:
                    print(f"{len(outliers)} valeurs abérrantes trouvées dans la colonne '{col}'")
                    df = df[~((df[col] < lower_bound) | (df[col] > upper_bound))]

            print(f"shape: {df.shape}\n")

print(cleaned_dataframes.keys())
